<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
المؤلف: [إيجور بولوسماك](https://www.linkedin.com/in/egor-polusmak/). تمت الترجمة والتحرير بواسطة [يوانيوان باو](https://www.linkedin.com/in/yuanyuanpao/). تخضع هذه المادة لشروط وأحكام ترخيص [Creative Commons CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). الاستخدام المجاني مسموح به لأي غرض غير تجاري.



# <center>الموضوع 9. تحليل السلاسل الزمنية في Python</center>
## <center>الجزء الثاني. التنبؤ بالمستقبل مع Facebook Prophet</center>



يجد التنبؤ بالسلاسل الزمنية تطبيقًا واسعًا في تحليلات البيانات. هذه ليست سوى بعض التوقعات التي يمكن تصورها للاتجاهات المستقبلية التي قد تكون مفيدة:
- عدد الخوادم التي ستحتاجها الخدمة عبر الإنترنت في العام المقبل.
- الطلب على منتج البقالة في السوبر ماركت في يوم معين.
- سعر إغلاق الغد للأصل المالي القابل للتداول.
على سبيل المثال، يمكننا التنبؤ بأداء فريق ما ثم استخدامه كخط أساس: أولاً لتحديد الأهداف للفريق، ومن ثم قياس أداء الفريق الفعلي بالنسبة إلى خط الأساس.
هناك عدد لا بأس به من الطرق المختلفة للتنبؤ بالاتجاهات المستقبلية، على سبيل المثال، [ARIMA](https://en.wikipedia.org/wiki/Autoregressive_integrated_moving_average)، [ARCH](https://en.wikipedia.org/wiki/Autoregressive_conditional_heteroskedasticity)، [النماذج التراجعية](https://en.wikipedia.org/wiki/Autoregressive_model)، [الشبكات العصبية](https://medium.com/machine-learning-world/neural-networks-for-algorithmic-trading-1-2-correct-time-series-forecasting-backtesting-9776bfd9e589).
في هذه المقالة، سنلقي نظرة على [Prophet](https://facebook.github.io/prophet/)، وهي مكتبة للتنبؤ بالسلاسل الزمنية أصدرها فيسبوك ومفتوحة المصدر في 23 فبراير 2017. وسنجربها أيضًا في مشكلة التنبؤ بالعدد اليومي للمشاركات المنشورة على Medium.



## الخطوط العريضة للمادة1. مقدمة
2. نموذج التنبؤ النبوي
3. الممارسة مع النبي
    * 3.1 التثبيت في بايثون
    * 3.2 مجموعة البيانات
    * 3.3 التحليل البصري الاستكشافي
    * 3.4 عمل توقعات
    * 3.5 تقييم الجودة المتوقعة
    * 3.6 التصور
4. تحول بوكس كوكس
5. ملخص
6. المراجع



## 1. مقدمة
وفقًا لـ [المقالة](https://research.fb.com/prophet-forecasting-at-scale/) في Facebook Research، تم تطوير Prophet في البداية بغرض إنشاء تنبؤات أعمال عالية الجودة. تحاول هذه المكتبة معالجة الصعوبات التالية الشائعة في العديد من سلاسل الأعمال الزمنية:
- التأثيرات الموسمية الناجمة عن سلوك الإنسان: الدورات الأسبوعية والشهرية والسنوية، والانخفاضات والقمم في أيام العطل الرسمية.
- التغيرات في الاتجاه بسبب المنتجات الجديدة وأحداث السوق.
- القيم المتطرفة.
ويدعي المؤلفون أنه حتى مع الإعدادات الافتراضية، في كثير من الحالات، تنتج مكتبتهم توقعات دقيقة مثل تلك التي يقدمها المحللون ذوو الخبرة.
علاوة على ذلك، يمتلك النبي عددًا من التخصيصات البديهية وسهلة التفسير والتي تسمح بتحسين جودة نموذج التنبؤ تدريجيًا. ما هو مهم بشكل خاص هو أن هذه المعلمات مفهومة تمامًا حتى بالنسبة لغير الخبراء في تحليل السلاسل الزمنية، وهو مجال من علوم البيانات يتطلب مهارة وخبرة معينة.بالمناسبة، المقال الأصلي يسمى "التنبؤ على نطاق واسع"، لكنه لا يتعلق بالحجم بالمعنى "المعتاد"، الذي يتناول المشاكل الحسابية والبنية التحتية لعدد كبير من برامج العمل. وفقا للمؤلفين، ينبغي للنبي أن يتوسع بشكل جيد في المجالات الثلاثة التالية:
- إمكانية الوصول إلى جمهور واسع من المحللين، ربما دون خبرة عميقة في السلاسل الزمنية.
- قابلية التطبيق على مجموعة واسعة من مشاكل التنبؤ المتميزة.
- تقدير الأداء الآلي لعدد كبير من التوقعات بما في ذلك وضع علامة على المشاكل المحتملة لفحصها لاحقا من قبل المحلل.



## 2. نموذج التنبؤ النبوي
الآن، دعونا نلقي نظرة فاحصة على كيفية عمل النبي. في جوهرها، تستخدم هذه المكتبة [نموذج الانحدار الإضافي](https://en.wikipedia.org/wiki/Additive_model) $y(t)$ الذي يشتمل على المكونات التالية:
$$y(t) = g(t) + s(t) + h(t) + \epsilon_{t},$$
حيث:
* الاتجاه $g(t)$ نماذج التغييرات غير الدورية.
* الموسمية $s(t)$ تمثل التغييرات الدورية.
* مكون العطلات $h(t)$ يساهم بمعلومات حول العطلات والمناسبات.
أدناه، سننظر في بعض الخصائص الهامة لمكونات النموذج هذه.



### الاتجاه
تطبق مكتبة النبي نموذجين محتملين للاتجاه لـ $g(t)$.
الأول يسمى *النمو المشبع غير الخطي*. ويتم تمثيله في شكل [نموذج النمو اللوجستي](https://en.wikipedia.org/wiki/Logistic_function):
$$g(t) = \frac{C}{1+e^{-k(t - m)}},$$
حيث:
* $C$ هي القدرة الاستيعابية (وهي القيمة القصوى للمنحنى).
* $k$ هو معدل النمو (الذي يمثل "انحدار" المنحنى).
* $m$ عبارة عن معلمة إزاحة.تسمح هذه المعادلة اللوجستية بنمذجة النمو غير الخطي مع التشبع، أي عندما يتناقص معدل نمو القيمة مع نموها. أحد الأمثلة النموذجية هو تمثيل نمو جمهور التطبيق أو موقع الويب.
في الواقع، $C$ و$k$ ليست بالضرورة ثوابت وقد تختلف بمرور الوقت. يدعم النبي الضبط التلقائي واليدوي لتنوعها. يمكن للمكتبة بنفسها اختيار النقاط المثلى لتغيرات الاتجاه من خلال ملاءمة البيانات التاريخية المقدمة. 
كما يسمح Prophet للمحللين بتعيين نقاط التغيير لمعدل النمو وقيم السعة يدويًا في نقاط زمنية مختلفة. على سبيل المثال، قد يكون لدى المحللين رؤى حول تواريخ الإصدارات السابقة التي أثرت بشكل بارز على بعض مؤشرات المنتجات الرئيسية.
نموذج الاتجاه الثاني هو *نموذج خطي مجزأ بسيط* مع معدل نمو ثابت. هو الأنسب للمشاكل دون نمو مشبع.



### الموسمية
يوفر المكون الموسمي $s(t)$ نموذجًا مرنًا للتغيرات الدورية بسبب الموسمية الأسبوعية والسنوية.
يتم تصميم البيانات الموسمية الأسبوعية باستخدام متغيرات وهمية. تمت إضافة ستة متغيرات جديدة: `monday`، `tuesday`، `wednesday`، `thursday`، `friday`، `saturday`، والتي تأخذ القيم 0 أو 1 حسب يوم الأسبوع. لم تتم إضافة الميزة `sunday` لأنها ستكون مزيجًا خطيًا من الأيام الأخرى من الأسبوع، وسيكون لهذه الحقيقة تأثير سلبي على النموذج.
يعتمد النموذج الموسمي السنوي في النبي على سلسلة فورييه.
منذ [الإصدار 0.2](https://github.com/facebook/prophet) يمكنك أيضًا استخدام *سلاسل زمنية فرعية* وإجراء *تنبؤات شبه يومية* بالإضافة إلى استخدام ميزة *الموسمية اليومية* الجديدة.



### الأعياد والمناسباتيمثل المكون $h(t)$ الأيام غير الطبيعية التي يمكن التنبؤ بها في العام بما في ذلك تلك الموجودة في جداول زمنية غير منتظمة، على سبيل المثال، أيام الجمعة السوداء.
للاستفادة من هذه الميزة، يحتاج المحلل إلى تقديم قائمة مخصصة بالأحداث.



### خطأ
يمثل مصطلح الخطأ $\epsilon(t)$ المعلومات التي لم تنعكس في النموذج. وعادة ما يتم تصميمه على أنه ضوضاء موزعة بشكل طبيعي.



### قياس النبي
للحصول على وصف تفصيلي للنموذج والخوارزميات المستخدمة في تطبيق Prophet، راجع المقالة ["التنبؤ على نطاق واسع"](https://peerj.com/preprints/3190/) بقلم Sean J. Taylor وBenjamin Letham.
قام المؤلفون أيضًا بمقارنة مكتبتهم بعدة طرق أخرى للتنبؤ بالسلاسل الزمنية. استخدموا [متوسط ​​النسبة المئوية للخطأ المطلق (MAPE)](https://en.wikipedia.org/wiki/Mean_absolute_percentage_error) كمقياس لدقة التنبؤ. في هذا البحث، أظهر النبي خطأ تنبؤي أقل بكثير من النماذج الأخرى.



<img src="../../img/topic9_benchmarking_prophet.png" />



دعونا نلقي نظرة فاحصة على كيفية قياس جودة التنبؤ في المقالة. للقيام بذلك، سنحتاج إلى صيغة متوسط ​​النسبة المئوية للخطأ المطلق.
اجعل $y_{i}$ هي *القيمة الفعلية (التاريخية)* و$\hat{y}_{i}$ هي *القيمة المتوقعة* التي يقدمها نموذجنا.
ثم $e_{i} = y_{i} - \hat{y}_{i}$ هو *خطأ التنبؤ* و$p_{i} =\frac{\displaystyle e_{i}}{\displaystyle y_{i}}$ هو *خطأ التنبؤ النسبي*.
نحن نحدد
$$MAPE = mean\big(\left |p_{i} \right |\big)$$
يستخدم MAPE على نطاق واسع كمقياس لدقة التنبؤ لأنه يعبر عن الخطأ كنسبة مئوية وبالتالي يمكن استخدامه في تقييمات النماذج على مجموعات بيانات مختلفة.
بالإضافة إلى ذلك، عند تقييم خوارزمية التنبؤ، قد يكون من المفيد حساب [MAE (متوسط ​​الخطأ المطلق)](https://en.wikipedia.org/wiki/Mean_absolute_error) للحصول على صورة للأخطاء بالأرقام المطلقة. باستخدام المكونات المحددة سابقا، ستكون معادلتها
$$MAE = mean\big(\left |e_{i}\right |\big)$$


بضع كلمات عن الخوارزميات التي تمت مقارنة النبي بها. معظمها بسيط جدًا وغالبًا ما يتم استخدامه كخط أساس لنماذج أخرى:
* `naive` هو أسلوب تنبؤي مبسط حيث نتوقع جميع القيم المستقبلية بالاعتماد فقط على الملاحظة في آخر نقطة زمنية متاحة.
* `snaive` (الموسمي الساذج) هو نموذج يقوم بتنبؤات ثابتة مع مراعاة المعلومات حول الموسمية. على سبيل المثال، في حالة البيانات الموسمية الأسبوعية لكل يوم اثنين مستقبلي، يمكننا التنبؤ بالقيمة من يوم الاثنين الأخير، وبالنسبة لجميع أيام الثلاثاء المقبلة، سنستخدم القيمة من يوم الثلاثاء الأخير وما إلى ذلك.
* `mean` يستخدم متوسط ​​قيمة البيانات كتنبؤ.
* `arima` يرمز إلى *المتوسط ​​المتحرك المتكامل ذاتي الانحدار*، راجع [ويكيبيديا](https://en.wikipedia.org/wiki/Autoregressive_integrated_moving_average) للحصول على التفاصيل.
* `ets` يرمز إلى *التجانس الأسي*، راجع [ويكيبيديا](https://en.wikipedia.org/wiki/Exponential_smoothing) للمزيد.



## 3. تدرب مع فيسبوك النبي
### 3.1 التثبيت في بايثون
أولا، تحتاج إلى تثبيت المكتبة. Prophet متاح لـ Python وR. وسيعتمد الاختيار على تفضيلاتك الشخصية ومتطلبات المشروع. علاوة على ذلك في هذه المقالة سوف نستخدم بايثون.
في بايثون يمكنك تثبيت Prophet باستخدام PyPI:
```
$ pip install fbprophet
```
في R يمكنك العثور على حزمة CRAN المقابلة. راجع [الوثائق](https://facebookincubator.github.io/prophet/docs/installation.html) للحصول على التفاصيل.
لنقم باستيراد الوحدات التي سنحتاجها، وتهيئة بيئتنا:


In [ ]:
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

%matplotlib inline


### 3.2 مجموعة البيانات
سنتوقع العدد اليومي للمشاركات المنشورة على [المتوسط](https://medium.com/).
أولاً، نقوم بتحميل مجموعة البيانات الخاصة بنا.


In [ ]:
df = pd.read_csv("../../data/medium_posts.csv.zip", sep="\t")

بعد ذلك، نترك جميع الأعمدة باستثناء `published` و`url`. يتوافق الأول مع البعد الزمني بينما يحدد الأخير المنشور بشكل فريد من خلال عنوان URL الخاص به. على طول الطريق نتخلص من التكرارات المحتملة والقيم المفقودة في البيانات:


In [ ]:
df = df[["published", "url"]].dropna().drop_duplicates()


بعد ذلك، نحتاج إلى تحويل `published` إلى تنسيق التاريخ والوقت لأن `pandas` يتعامل مع هذا الحقل كقيمة سلسلة بشكل افتراضي.


In [ ]:
df["published"] = pd.to_datetime(df["published"])


دعونا نفرز إطار البيانات حسب الوقت ونلقي نظرة على ما لدينا:


In [ ]:
df.sort_values(by=["published"]).head(n=3)


كان تاريخ الإصدار العام لـ Medium هو 15 أغسطس 2012. ولكن، كما ترون من البيانات أعلاه، هناك عدة صفوف على الأقل بتواريخ نشر سابقة بكثير. لقد ظهرت بطريقة أو بأخرى في مجموعة البيانات لدينا، لكنها ليست شرعية على الإطلاق. سنقوم فقط بقص سلسلتنا الزمنية للاحتفاظ فقط بالصفوف التي تقع في الفترة من 15 أغسطس 2012 إلى 25 يونيو 2017:


In [ ]:
df = df[
    (df["published"] > "2012-08-15") & (df["published"] < "2017-06-26")
].sort_values(by=["published"])
df.head(n=3)

In [ ]:
df.tail(n=3)


نظرًا لأننا سنقوم بالتنبؤ بعدد المنشورات المنشورة، فسوف نقوم بتجميع وإحصاء المشاركات الفريدة في كل وقت محدد. سنقوم بتسمية العمود الجديد المقابل `posts`:


In [ ]:
aggr_df = df.groupby("published")[["url"]].count()
aggr_df.columns = ["posts"]


في هذه الممارسة، نحن مهتمون بعدد المشاركات **في اليوم**. لكن في هذه اللحظة جميع بياناتنا مقسمة إلى فترات زمنية غير منتظمة تقل عن يوم واحد. وهذا ما يسمى *سلسلة زمنية شبه يومية*. لرؤيتها، دعونا نطبع الصفوف الثلاثة الأولى:


In [ ]:
aggr_df.head(n=3)


لإصلاح هذه المشكلة، نحتاج إلى تجميع عدد المشاركات حسب "سلال" بحجم التاريخ. في تحليل السلاسل الزمنية، يشار إلى هذه العملية باسم *إعادة التشكيل*. وإذا *قللنا* معدل أخذ عينات البيانات، فغالبًا ما يُطلق على ذلك اسم *الاختزال*.
لحسن الحظ، `pandas` لديه وظيفة مضمنة لهذه المهمة. سنقوم بإعادة تشكيل مؤشر الوقت الخاص بنا وصولاً إلى صناديق اليوم الواحد:


In [ ]:
daily_df = aggr_df.resample("D").apply(sum)
daily_df.head(n=3)


### 3.3 التحليل البصري الاستكشافيكما هو الحال دائمًا، قد يكون من المفيد والمفيد النظر إلى تمثيل رسومي لبياناتك.
سوف نقوم بإنشاء مؤامرة سلسلة زمنية لكامل النطاق الزمني. إن عرض البيانات على مدى هذه الفترة الطويلة من الزمن يمكن أن يعطي أدلة حول الموسمية والانحرافات غير الطبيعية الواضحة.
أولاً، نقوم باستيراد وتهيئة مكتبة `Plotly`، والتي تسمح بإنشاء مخططات تفاعلية جميلة:


In [ ]:
from plotly import graph_objs as go
from plotly.offline import init_notebook_mode, iplot

# Initialize plotly
init_notebook_mode(connected=True)


نحدد أيضًا وظيفة مساعدة، والتي سترسم إطارات البيانات الخاصة بنا طوال المقالة:


In [ ]:
def plotly_df(df, title=""):
    """Visualize all the dataframe columns as line plots."""
    common_kw = dict(x=df.index, mode="lines")
    data = [go.Scatter(y=df[c], name=c, **common_kw) for c in df.columns]
    layout = dict(title=title)
    fig = dict(data=data, layout=layout)
    iplot(fig, show_link=False)


دعونا نحاول رسم مجموعة البيانات الخاصة بنا *كما هي*:


In [ ]:
plotly_df(daily_df, title="Posts on Medium (daily)")


قد يكون من الصعب تحليل البيانات عالية التردد. حتى مع إمكانية التكبير التي يوفرها `Plotly`، فمن الصعب استنتاج أي شيء ذي معنى من هذا الرسم البياني باستثناء الاتجاه الصعودي البارز والمتسارع.
لتقليل التشويش، سنعيد تشكيل العد التنازلي للمنشورات إلى الصناديق الأسبوعية. إلى جانب *binning*، تتضمن التقنيات الأخرى الممكنة لتقليل الضوضاء [Moving-Average Smoothing](https://en.wikipedia.org/wiki/Moving_average) و[التجانس الأسي](https://en.wikipedia.org/wiki/Exponential_smoothing)، من بين تقنيات أخرى.
نقوم بحفظ إطار البيانات المصغر الخاص بنا في متغير منفصل لأننا في هذه الممارسة سنعمل فقط مع السلاسل اليومية:


In [ ]:
weekly_df = daily_df.resample("W").apply(sum)


وأخيراً نرسم النتيجة:


In [ ]:
plotly_df(weekly_df, title="Posts on Medium (weekly)")


يثبت هذا الرسم البياني المصغر أنه أفضل إلى حد ما بالنسبة لتصور المحلل.
إحدى الوظائف الأكثر فائدة التي يوفرها `Plotly` هي القدرة على التعمق بسرعة في فترات زمنية مختلفة من أجل فهم البيانات بشكل أفضل والعثور على أدلة مرئية حول الاتجاهات المحتملة والتأثيرات الدورية وغير المنتظمة. 
على سبيل المثال، يُظهر لنا تكبير بضع سنوات متتالية نقاطًا زمنية تتوافق مع عطلة عيد الميلاد، والتي تؤثر بشكل كبير على سلوكيات الإنسان.الآن، سنقوم بحذف السنوات القليلة الأولى من الملاحظات، حتى عام 2015. أولاً، لن تساهم كثيرًا في جودة التنبؤ في عام 2017. ثانيًا، من المرجح أن تؤدي هذه السنوات الأولى، التي تحتوي على عدد منخفض جدًا من المشاركات يوميًا، إلى زيادة التشويش في توقعاتنا، حيث سيضطر النموذج إلى ملاءمة هذه البيانات التاريخية غير الطبيعية مع بيانات أكثر صلة وإرشادية من السنوات الأخيرة.


In [ ]:
daily_df = daily_df.loc[daily_df.index >= "2015-01-01"]
daily_df.head(n=3)


خلاصة القول، من التحليل البصري، يمكننا أن نرى أن مجموعة البيانات لدينا غير ثابتة مع اتجاه متزايد بارز. كما يوضح أيضًا الموسمية الأسبوعية والسنوية وعدد الأيام غير الطبيعية في كل عام.



### 3.4 عمل توقعات
واجهة برمجة تطبيقات Prophet تشبه إلى حد كبير تلك التي يمكنك العثور عليها في `sklearn`. نقوم أولاً بإنشاء نموذج، ثم نستدعي الطريقة `fit`، وأخيرًا نقوم بالتنبؤ. الإدخال إلى الأسلوب `fit` هو `DataFrame` مع عمودين:
* `ds` (ختم التاريخ) يجب أن يكون من النوع `date` أو `datetime`.
* `y` هي قيمة رقمية نريد التنبؤ بها.
للبدء، سنقوم باستيراد المكتبة وتجاهل رسائل التشخيص غير المهمة:


In [ ]:
import logging

from fbprophet import Prophet

logging.getLogger().setLevel(logging.ERROR)


دعونا نحول إطار البيانات الخاص بنا إلى التنسيق الذي يطلبه النبي:


In [ ]:
df = daily_df.reset_index()
df.columns = ["ds", "y"]
# converting timezones (issue https://github.com/facebook/prophet/issues/831)
df["ds"] = df["ds"].dt.tz_convert(None)
df.tail(n=3)


ينصح مؤلفو المكتبة عمومًا بوضع تنبؤات بناءً على عدة أشهر على الأقل، ومن الناحية المثالية، أكثر من عام من البيانات التاريخية. ولحسن الحظ، في حالتنا لدينا أكثر من عامين من البيانات لتناسب النموذج.
لقياس جودة توقعاتنا، نحتاج إلى تقسيم مجموعة البيانات الخاصة بنا إلى *الجزء التاريخي*، وهو الشريحة الأولى والأكبر من بياناتنا، و *جزء التنبؤ*، والذي سيكون موجودًا في نهاية المخطط الزمني. سنقوم بإزالة الشهر الأخير من مجموعة البيانات لاستخدامه لاحقًا كهدف للتنبؤ:


In [ ]:
prediction_size = 30
train_df = df[:-prediction_size]
train_df.tail(n=3)

نحتاج الآن إلى إنشاء كائن `Prophet` جديد. هنا يمكننا تمرير معلمات النموذج إلى المُنشئ. ولكن في هذه المقالة سوف نستخدم الإعدادات الافتراضية. ثم نقوم بتدريب نموذجنا من خلال استدعاء أسلوب `fit` في مجموعة بيانات التدريب الخاصة بنا:


In [ ]:
m = Prophet()
m.fit(train_df);


باستخدام الطريقة المساعدة `Prophet.make_future_dataframe`، نقوم بإنشاء إطار بيانات يحتوي على جميع التواريخ من السجل ويمتد أيضًا إلى المستقبل لتلك الأيام الثلاثين التي تركناها من قبل.


In [ ]:
future = m.make_future_dataframe(periods=prediction_size)
future.tail(n=3)


نحن نتوقع القيم باستخدام `Prophet` عن طريق تمرير التواريخ التي نريد إنشاء توقعات لها. إذا قمنا أيضًا بتوفير التواريخ التاريخية (كما في حالتنا)، فبالإضافة إلى التنبؤ، سنحصل على عينة مناسبة للتاريخ. لنستدعي طريقة `predict` الخاصة بالنموذج باستخدام إطار البيانات `future` الخاص بنا كمدخل:


In [ ]:
forecast = m.predict(future)
forecast.tail(n=3)


في إطار البيانات الناتج، يمكنك رؤية العديد من الأعمدة التي تميز التنبؤ، بما في ذلك مكونات الاتجاه والموسمية بالإضافة إلى فترات الثقة الخاصة بها. يتم تخزين التوقعات نفسها في العمود `yhat`.
تحتوي مكتبة النبي على أدوات التصور المدمجة الخاصة بها والتي تمكننا من تقييم النتيجة بسرعة.
أولاً، هناك طريقة تسمى `Prophet.plot` ترسم جميع النقاط من التوقعات:


In [ ]:
m.plot(forecast);


لا يبدو هذا الرسم البياني مفيدًا للغاية. الاستنتاج النهائي الوحيد الذي يمكننا استخلاصه هنا هو أن النموذج تعامل مع العديد من نقاط البيانات على أنها قيم متطرفة.
قد تكون الوظيفة الثانية `Prophet.plot_components` أكثر فائدة في حالتنا. فهو يسمح لنا بمراقبة المكونات المختلفة للنموذج بشكل منفصل: الاتجاه، والموسمية السنوية والأسبوعية. بالإضافة إلى ذلك، إذا قمت بتوفير معلومات حول العطلات والمناسبات إلى النموذج الخاص بك، فسيتم عرضها أيضًا في هذه المؤامرة.
دعونا نجربها:


In [ ]:
m.plot_components(forecast);

كما ترون من الرسم البياني للاتجاهات، قام النبي بعمل جيد من خلال ملاءمة النمو المتسارع للمشاركات الجديدة في نهاية عام 2016. ويؤدي الرسم البياني للموسمية الأسبوعية إلى استنتاج مفاده أن عدد المنشورات الجديدة في أيام السبت والأحد أقل عادةً من الأيام الأخرى من الأسبوع. في الرسم البياني الموسمي السنوي هناك انخفاض ملحوظ في يوم عيد الميلاد.



### 3.5 تقييم الجودة المتوقع



دعونا نقيم جودة الخوارزمية من خلال حساب مقاييس الخطأ لآخر 30 يومًا التي توقعناها. لهذا، سنحتاج إلى الملاحظات $y_i$ والقيم المتوقعة المقابلة $\hat{y}_i$.
دعونا نلقي نظرة على الكائن `forecast` الذي أنشأته المكتبة لنا:


In [ ]:
print(", ".join(forecast.columns))


يمكننا أن نرى أن إطار البيانات هذا يحتوي على جميع المعلومات التي نحتاجها باستثناء القيم التاريخية. نحتاج إلى ربط الكائن `forecast` بالقيم الفعلية `y` من مجموعة البيانات الأصلية `df`. لهذا سوف نحدد وظيفة مساعدة سنعيد استخدامها لاحقًا:


In [ ]:
def make_comparison_dataframe(historical, forecast):
    """Join the history with the forecast.
    
       The resulting dataset will contain columns 'yhat', 'yhat_lower', 'yhat_upper' and 'y'.
    """
    return forecast.set_index("ds")[["yhat", "yhat_lower", "yhat_upper"]].join(
        historical.set_index("ds")
    )


دعونا نطبق هذه الوظيفة على توقعاتنا الأخيرة:


In [ ]:
cmp_df = make_comparison_dataframe(df, forecast)
cmp_df.tail(n=3)


سنقوم أيضًا بتعريف وظيفة مساعدة سنستخدمها لقياس جودة توقعاتنا باستخدام مقاييس الخطأ MAPE وMAE:


In [ ]:
def calculate_forecast_errors(df, prediction_size):
    """Calculate MAPE and MAE of the forecast.
    
       Args:
           df: joined dataset with 'y' and 'yhat' columns.
           prediction_size: number of days at the end to predict.
    """

    # Make a copy
    df = df.copy()

    # Now we calculate the values of e_i and p_i according to the formulas given in the article above.
    df["e"] = df["y"] - df["yhat"]
    df["p"] = 100 * df["e"] / df["y"]

    # Recall that we held out the values of the last `prediction_size` days
    # in order to predict them and measure the quality of the model.

    # Now cut out the part of the data which we made our prediction for.
    predicted_part = df[-prediction_size:]

    # Define the function that averages absolute error values over the predicted part.
    error_mean = lambda error_name: np.mean(np.abs(predicted_part[error_name]))

    # Now we can calculate MAPE and MAE and return the resulting dictionary of errors.
    return {"MAPE": error_mean("p"), "MAE": error_mean("e")}


دعونا نستخدم وظيفتنا:


In [ ]:
for err_name, err_value in calculate_forecast_errors(cmp_df, prediction_size).items():
    print(err_name, err_value)


ونتيجة لذلك، يبلغ الخطأ النسبي لتوقعاتنا (MAPE) حوالي 22.6%، وفي المتوسط يكون نموذجنا خاطئًا بحوالي 70 مشاركة (MAE).



### 3.6 التصور
دعونا نخلق تصورنا الخاص للنموذج الذي بناه النبي. وسوف تشمل القيم الفعلية والتنبؤات وفترات الثقة.أولاً، سنقوم بتخطيط البيانات لفترة زمنية أقصر لتسهيل التمييز بين نقاط البيانات. ثانيًا، سنعرض أداء النموذج فقط للفترة التي توقعناها، وهي آخر 30 يومًا. يبدو أن هذين الإجراءين يجب أن يمنحونا حبكة أكثر وضوحًا.
ثالثًا، سوف نستخدم `Plotly` لجعل مخططنا تفاعليًا، وهو أمر رائع للاستكشاف.
سنحدد وظيفة مساعدة مخصصة `show_forecast` ونطلق عليها اسم (لمزيد من المعلومات حول كيفية عملها، يرجى الرجوع إلى التعليقات في الكود و[الوثائق](https://plot.ly/python/)):


In [ ]:
def show_forecast(cmp_df, num_predictions, num_values, title):
    """Visualize the forecast."""

    def create_go(name, column, num, **kwargs):
        points = cmp_df.tail(num)
        args = dict(name=name, x=points.index, y=points[column], mode="lines")
        args.update(kwargs)
        return go.Scatter(**args)

    lower_bound = create_go(
        "Lower Bound",
        "yhat_lower",
        num_predictions,
        line=dict(width=0),
        marker=dict(color="gray"),
    )
    upper_bound = create_go(
        "Upper Bound",
        "yhat_upper",
        num_predictions,
        line=dict(width=0),
        marker=dict(color="gray"),
        fillcolor="rgba(68, 68, 68, 0.3)",
        fill="tonexty",
    )
    forecast = create_go(
        "Forecast", "yhat", num_predictions, line=dict(color="rgb(31, 119, 180)")
    )
    actual = create_go("Actual", "y", num_values, marker=dict(color="red"))

    # In this case the order of the series is important because of the filling
    data = [lower_bound, upper_bound, forecast, actual]

    layout = go.Layout(yaxis=dict(title="Posts"), title=title, showlegend=False)
    fig = go.Figure(data=data, layout=layout)
    iplot(fig, show_link=False)


show_forecast(cmp_df, prediction_size, 100, "New posts on Medium")


للوهلة الأولى، يبدو التنبؤ بالقيم المتوسطة من خلال نموذجنا معقولا. يمكن تفسير القيمة العالية لـ MAPE التي ذكرناها أعلاه من خلال حقيقة أن النموذج فشل في اللحاق بزيادة سعة الذروة إلى الذروة ذات الموسمية الضعيفة.
كما يمكننا أن نستنتج من الرسم البياني أعلاه أن العديد من القيم الفعلية تقع خارج فترة الثقة. قد لا يكون النبي مناسبًا للسلاسل الزمنية ذات التباين غير المستقر، على الأقل عند استخدام الإعدادات الافتراضية. سنحاول إصلاح ذلك عن طريق تطبيق تحويل على بياناتنا.



## 4. تحول بوكس-كوكس



لقد استخدمنا النبي حتى الآن مع الإعدادات الافتراضية والبيانات الأصلية. سنترك معلمات النموذج وحدها. ولكن على الرغم من هذا لا يزال لدينا بعض المجال للتحسين. في هذا القسم، سوف نقوم بتطبيق [تحويل Box–Cox](http://onlinestatbook.com/2/transformations/box-cox.html) على سلسلتنا الأصلية. دعونا نرى أين سيقودنا.
بضع كلمات حول هذا التحول. هذا تحويل بيانات رتيب يمكن استخدامه لتحقيق الاستقرار في التباين. سوف نستخدم تحويل Box–Cox ذو المعلمة الواحدة، والذي يتم تعريفه بالتعبير التالي:
$$
\begin{equation}
  boxcox^{(\lambda)}(y_{i}) = \begin{cases}
    \frac{\displaystyle y_{i}^{\lambda} - 1}{\displaystyle \lambda} &, \text{if $\lambda \neq 0$}.\\
    ln(y_{i}) &, \text{if $\lambda = 0$}.
  \end{cases}
\end{equation}
$$سنحتاج إلى تنفيذ عكس هذه الوظيفة حتى نتمكن من استعادة مقياس البيانات الأصلي. من السهل أن نرى أن المعكوس يتم تعريفه على النحو التالي:
$$
\begin{equation}
  invboxcox^{(\lambda)}(y_{i}) = \begin{cases}
    e^{\left (\frac{\displaystyle ln(\lambda y_{i} + 1)}{\displaystyle \lambda} \right )} &, \text{if $\lambda \neq 0$}.\\
    e^{y_{i}} &, \text{if $\lambda = 0$}.
  \end{cases}
\end{equation}
$$
يتم تنفيذ الوظيفة المقابلة في بايثون على النحو التالي:


In [ ]:
def inverse_boxcox(y, lambda_):
    return np.exp(y) if lambda_ == 0 else np.exp(np.log(lambda_ * y + 1) / lambda_)


أولاً، نقوم بإعداد مجموعة البيانات الخاصة بنا عن طريق تحديد فهرسها:


In [ ]:
train_df2 = train_df.copy().set_index("ds")


بعد ذلك، نطبق الدالة `stats.boxcox` من `Scipy`، والتي تطبق تحويل Box–Cox. في حالتنا سوف يعود قيمتين. الأول هو السلسلة المحولة والثاني هو القيمة التي تم العثور عليها لـ $\lambda$ والتي تعتبر مثالية من حيث الحد الأقصى لاحتمالية السجل:


In [ ]:
train_df2["y"], lambda_prophet = stats.boxcox(train_df2["y"])
train_df2.reset_index(inplace=True)


نقوم بإنشاء نموذج `Prophet` جديد ونكرر دورة التنبؤ الملائمة التي قمنا بها بالفعل أعلاه:


In [ ]:
m2 = Prophet()
m2.fit(train_df2)
future2 = m2.make_future_dataframe(periods=prediction_size)
forecast2 = m2.predict(future2)


في هذه المرحلة، نحتاج إلى عكس تحويل Box–Cox باستخدام الدالة العكسية والقيمة المعروفة $\lambda$:


In [ ]:
for column in ["yhat", "yhat_lower", "yhat_upper"]:
    forecast2[column] = inverse_boxcox(forecast2[column], lambda_prophet)


سنقوم هنا بإعادة استخدام أدواتنا لإنشاء إطار بيانات المقارنة وحساب الأخطاء:


In [ ]:
cmp_df2 = make_comparison_dataframe(df, forecast2)
for err_name, err_value in calculate_forecast_errors(cmp_df2, prediction_size).items():
    print(err_name, err_value)


لذلك، يمكننا بالتأكيد أن نذكر زيادة في جودة النموذج.
وأخيرًا، دعونا نرسم أداءنا السابق مع أحدث النتائج جنبًا إلى جنب. لاحظ أننا نستخدم `prediction_size` للمعلمة الثالثة لتكبير الفاصل الزمني المتوقع:


In [ ]:
show_forecast(cmp_df, prediction_size, 100, "No transformations")
show_forecast(cmp_df2, prediction_size, 100, "Box–Cox transformation")


نرى أن توقعات التغيرات الأسبوعية في الرسم البياني الثاني أقرب بكثير إلى القيم الحقيقية الآن.



## 5. ملخص



لقد ألقينا نظرة على *Prophet*، وهي مكتبة توقعات مفتوحة المصدر تستهدف بشكل خاص السلاسل الزمنية للأعمال. لقد قمنا أيضًا ببعض التدريب العملي على التنبؤ بالسلاسل الزمنية.وكما رأينا فإن المكتبة النبوية لا تصنع العجائب، وتوقعاتها خارج الصندوق ليست [مثالية](https://en.wikipedia.org/wiki/No_free_lunch_in_search_and_optimization). لا يزال الأمر متروكًا لعالم البيانات لاستكشاف نتائج التوقعات وضبط معلمات النموذج وتحويل البيانات عند الضرورة.
ومع ذلك، هذه المكتبة سهلة الاستخدام وقابلة للتخصيص بسهولة. إن القدرة الوحيدة على مراعاة الأيام غير الطبيعية المعروفة للمحلل مسبقًا قد تحدث فرقًا في بعض الحالات.
وبشكل عام، تستحق مكتبة النبي أن تكون جزءًا من صندوق أدواتك التحليلية.



## 6. المراجع



- [مستودع النبي] الرسمي (https://github.com/facebookincubator/prophet) على GitHub.
- [التوثيق النبوي] الرسمي (https://facebookincubator.github.io/prophet/docs/quick_start.html).
- Sean J. Taylor, Benjamin Letham ["التنبؤ على نطاق واسع"](https://facebookincubator.github.io/prophet/static/prophet_paper_20170113.pdf) - ورقة علمية تشرح الخوارزمية التي تضع أساس `Prophet`.
- [التنبؤ بحركة مرور موقع الويب باستخدام مكتبة النبي على فيسبوك](http://pbpython.com/prophet-overview.html) - نظرة عامة `Prophet` مع مثال للتنبؤ بحركة مرور موقع الويب.
- Rob J. Hyndman, George Athanasopoulos ["التنبؤ: المبادئ والممارسة"](https://www.otexts.org/fpp) - كتاب جيد جدًا عبر الإنترنت حول التنبؤ بالسلاسل الزمنية.